In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output
from resources.cut_off_edges import cut_off_edges
from resources.standarize_image import standardize_image

In [2]:
project_dir = Path.cwd().parents[1]
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

data = np.load(
    project_dir / "data" / "processed_volumes.npz",
    allow_pickle=False,
)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
uids = data["uid"]

In [3]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
indices = np.arange(len(y))

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_train = X[train_indices]
X_test = X[test_indices]

y_train = y[train_indices]
y_test = y[test_indices]

uids_train = uids[train_indices]
uids_test = uids[test_indices]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (1089, 1, 50, 40, 35)
X_test: (273, 1, 50, 40, 35)


In [4]:
class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()
        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

In [5]:
train_dataset = VolumeDataset(X_train, y_train)
test_dataset = VolumeDataset(X_test, y_test)
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0,
)

validation_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

In [6]:
from torch import nn
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=8,
                out_channels=16,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = Simple3DCNN().to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

print("Urządzenie:", device)

Urządzenie: cpu


In [8]:
import copy

best_validation_loss = float("inf")
best_model_state = None

epochs = 500

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for volumes_batch, labels_batch in train_loader:
        volumes_batch = volumes_batch.to(device)
        labels_batch = labels_batch.to(device)

        optimizer.zero_grad()

        logits = model(volumes_batch)
        loss = criterion(logits, labels_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(labels_batch)

    train_loss /= len(train_dataset)

    model.eval()
    validation_loss = 0.0

    with torch.no_grad():
        for volumes_batch, labels_batch in validation_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            validation_loss += loss.item() * len(labels_batch)

    validation_loss /= len(test_dataset)

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"train: {train_loss:.4f} | "
        f"validation: {validation_loss:.4f}"
    )

model.load_state_dict(best_model_state)

print("Najlepszy validation loss:", best_validation_loss)

Epoka 1/500 | train: 0.6894 | validation: 0.6878
Epoka 2/500 | train: 0.6890 | validation: 0.6879
Epoka 3/500 | train: 0.6800 | validation: 0.6829
Epoka 4/500 | train: 0.6581 | validation: 0.5765
Epoka 5/500 | train: 0.6732 | validation: 0.6866
Epoka 6/500 | train: 0.6856 | validation: 0.6840
Epoka 7/500 | train: 0.6779 | validation: 0.6666
Epoka 8/500 | train: 0.6041 | validation: 0.5143
Epoka 9/500 | train: 0.5321 | validation: 0.4911
Epoka 10/500 | train: 0.5240 | validation: 0.4568
Epoka 11/500 | train: 0.5055 | validation: 0.4631
Epoka 12/500 | train: 0.4807 | validation: 0.4267
Epoka 13/500 | train: 0.4755 | validation: 0.4297
Epoka 14/500 | train: 0.4540 | validation: 0.3973
Epoka 15/500 | train: 0.4661 | validation: 0.3942
Epoka 16/500 | train: 0.4636 | validation: 0.4104
Epoka 17/500 | train: 0.4603 | validation: 0.4316
Epoka 18/500 | train: 0.4615 | validation: 0.4023
Epoka 19/500 | train: 0.4506 | validation: 0.3752
Epoka 20/500 | train: 0.4330 | validation: 0.4704
Epoka 21/

In [9]:
from pathlib import Path
from datetime import datetime
import torch


models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

model_path = models_dir / f"second_3d_cnn_{timestamp}.pth"

torch.save(
    model.state_dict(),
    model_path,
)

print(f"Zapisano model: {model_path}")

Zapisano model: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\models\second_3d_cnn_2026-09-03_19-39-16.pth
